# QIF Analyses (AOL treated dataset)

In [ ]:
import pandas, random
from bvmlib.bvm import BVM

## AOL treated dataset

In [ ]:
%%time
# Loads AOL-treated dataset into aol_treated DataFrame.
aol_treated = pandas.read_csv('AOL-treated.csv', low_memory=False, on_bad_lines='warn', index_col=0)

In [ ]:
%%time
aol_treated['QueryTime'] = pandas.to_datetime(aol_treated['QueryTime'])

In [ ]:
display(aol_treated)

## Drop singletons

In [ ]:
%%time
# Lists rows for RandID individuals with only one domain in their browsing history, to be dropped.
temp = aol_treated[['RandID','Domain']].groupby(['RandID']).nunique()
singletons = temp[temp['Domain'] == 1].index.to_list()
print(len(singletons))

In [ ]:
aol_treated = aol_treated[~aol_treated.RandID.isin(singletons)]

In [ ]:
aol_treated = aol_treated.reset_index(drop=True)

In [ ]:
display(aol_treated)

## Drop outlier

In [ ]:
%%time
# One individual has too many domains in their browsing history and is most probably a bot, to be dropped.
display(aol_treated['RandID'].value_counts())

In [ ]:
aol_treated = aol_treated.drop(aol_treated[aol_treated['RandID'] == 463921].index.to_list())

In [ ]:
aol_treated = aol_treated.reset_index(drop=True)

In [ ]:
display(aol_treated)

## Experiment 1: Third-party cookies on AOL treated dataset

### Define browsing histories

In [ ]:
%%time
aol_treated['BrowsingHistory'] = list(zip(aol_treated.Domain, aol_treated.QueryTime))

In [ ]:
display(aol_treated)

In [ ]:
%%time
aol_treated_browsing_history = aol_treated[['RandID','BrowsingHistory']].groupby('RandID', sort=False)[['BrowsingHistory']].agg(lambda x : list(x)).reset_index()

In [ ]:
%%time
aol_treated_browsing_history['BrowsingHistory'] = aol_treated_browsing_history['BrowsingHistory'].map(lambda x : str(x))

In [ ]:
display(aol_treated_browsing_history)

### Privacy

In [ ]:
temp = aol_treated_browsing_history.copy()
temp['UID'] = temp['RandID']
temp = temp.drop(columns=['BrowsingHistory'], inplace=False)
display(temp, temp.nunique())

In [ ]:
%%time
E1P = BVM(temp)
E1P.qids(['UID'])
E1P.sensitive(['RandID'])
results = E1P.assess()

In [ ]:
display(results['re_id'], results['att_inf'])

In [ ]:
# Prior Bayes vulnerability.
display(results['re_id']['Prior'][0], results['att_inf']['Prior'][0])

In [ ]:
# Posterior Bayes vulnerability.
display(results['re_id']['Posterior'][0], results['att_inf']['Posterior'][0])

In [ ]:
# Bayes leakage.
display(results['re_id']['Posterior'][0]/results['re_id']['Prior'][0], results['att_inf']['Posterior'][0]/results['att_inf']['Prior'][0])

### Utility

In [ ]:
temp = aol_treated_browsing_history.copy()
temp = temp.rename(columns={'RandID':'UID'}, inplace=False)
display(temp, temp.nunique())

In [ ]:
%%time
E1U = BVM(temp)
E1U.qids(['UID'])
E1U.sensitive(['BrowsingHistory'])
results = E1U.assess()

In [ ]:
display(results['re_id'], results['att_inf'])

In [ ]:
# Prior Bayes vulnerability.
display(results['re_id']['Prior'][0], results['att_inf']['Prior'][0])

In [ ]:
# Posterior Bayes vulnerability.
display(results['re_id']['Posterior'][0], results['att_inf']['Posterior'][0])

In [ ]:
# Bayes leakage.
display(results['re_id']['Posterior'][0]/results['re_id']['Prior'][0], results['att_inf']['Posterior'][0]/results['att_inf']['Prior'][0])